# Wind deep-learning baselines (Darts)

Main-result reproduction only: **MAE, RMSE, and aQL**. Supports loading existing checkpoints or tuning/training from scratch.


## 1. Colab environment



In [ ]:
!pip install -r requirements/darts.txt


In [ ]:
from pathlib import Path
import sys, numpy as np, pandas as pd, torch, optuna
REPO_ROOT=Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT/"requirements").exists(): REPO_ROOT=REPO_ROOT.parent
ROOT=REPO_ROOT/"wind"
if str(REPO_ROOT) not in sys.path: sys.path.insert(0,str(REPO_ROOT))
if str(ROOT/"src") not in sys.path: sys.path.insert(0,str(ROOT/"src"))
from preprocessing import load_wind, describe_wind, NWP_FEATURES, SEED
from src.metrics import print_metrics
from power_conversion import convert_wind_result_to_power
from sklearn import preprocessing
from pytorch_lightning.callbacks import EarlyStopping
from darts import TimeSeries, concatenate
from darts.dataprocessing.transformers import Scaler
from darts.metrics import rmse
from darts.models import TFTModel, TSMixerModel, RNNModel, TiDEModel, DLinearModel, NLinearModel
from darts.utils.likelihood_models import QuantileRegression
data=load_wind(ROOT/"data"/"wind")
print(describe_wind(data))


## 2. Configuration


In [ ]:
MODEL="tft"  # tft, tsmixer, deepar, tide, dlinear, nlinear
LOAD_EXISTING=True
SAVE_MODEL=True
N_TRIALS=30
BASE_QUANTILES=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]
TAIL_QUANTILES=[0.01,0.025,0.05,0.95,0.975,0.99]
OUTPUT_QUANTILES=sorted(set(BASE_QUANTILES+TAIL_QUANTILES))
QUANTILES=BASE_QUANTILES
MODEL_QUANTILES=OUTPUT_QUANTILES
NUM_SAMPLES=100
MODEL_DIR=ROOT/"models"/"darts"; MODEL_DIR.mkdir(parents=True,exist_ok=True)
OUTPUT_DIR=ROOT/"outputs"/"predictions"; OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
CHECKPOINT_FILES={"tft":"tft.darts","tsmixer":"tsmixer.darts","deepar":"deepar.darts","tide":"tide.darts",
                  "dlinear":"dlinear.darts","nlinear":"nlinear.darts"}
MODEL_CLASSES={"tft":TFTModel,"tsmixer":TSMixerModel,"deepar":RNNModel,"tide":TiDEModel,"dlinear":DLinearModel,"nlinear":NLinearModel}
MODEL_PATH=MODEL_DIR/CHECKPOINT_FILES[MODEL]
print("Mode:","LOAD" if LOAD_EXISTING else "TRAIN", "|", MODEL_PATH)


## 3. Data and training-only scaling


In [ ]:
series=TimeSeries.from_group_dataframe(data.full,time_col="timestamp",value_cols="target",group_cols="item_id",freq="h")
series=[s.astype(np.float32) for s in series]
covs=TimeSeries.from_group_dataframe(data.full,time_col="timestamp",value_cols=NWP_FEATURES,group_cols="item_id",freq="h")
covs=[c.astype(np.float32) for c in covs]
cutoff=pd.Timestamp("2020-03-14 23:00")
train,val,test=[],[],[]
for ts in series:
    tr,te=ts.split_after(cutoff); tr,va=tr.split_after(0.8); train.append(tr); val.append(va); test.append(te)
target_scaler=Scaler(preprocessing.StandardScaler())
train_t=target_scaler.fit_transform(train); val_t=target_scaler.transform(val); series_t=target_scaler.transform(series)
ctrain,cval=[],[]
for cov in covs:
    tr,_=cov.split_after(cutoff); tr,va=tr.split_after(0.8); ctrain.append(tr); cval.append(va)
cov_scaler=Scaler(preprocessing.StandardScaler())
ctrain_t=cov_scaler.fit_transform(ctrain); cval_t=cov_scaler.transform(cval); covs_t=cov_scaler.transform(covs)


## 4. Load existing model OR tune/train


In [ ]:
def early_stopper(): return EarlyStopping(monitor="val_loss",patience=5,min_delta=1e-3,mode="min",verbose=True)
def build_model(name,p,model_name):
    common=dict(output_chunk_length=24,batch_size=p["batch_size"],n_epochs=100,optimizer_kwargs={"lr":p["lr"]},random_state=SEED,likelihood=QuantileRegression(MODEL_QUANTILES),pl_trainer_kwargs={"callbacks":[early_stopper()],"accelerator":"auto","devices":1,"enable_checkpointing":True,"logger":False},model_name=model_name,force_reset=True,save_checkpoints=True)
    if name=="tft": return TFTModel(input_chunk_length=p["input_chunk_length"],hidden_size=p["hidden_size"],num_attention_heads=p["num_attention_heads"],lstm_layers=p["lstm_layers"],dropout=p["dropout"],**common)
    if name=="tsmixer": return TSMixerModel(input_chunk_length=p["input_chunk_length"],hidden_size=p["hidden_size"],dropout=p["dropout"],**common)
    if name=="deepar": return RNNModel(model="LSTM",hidden_dim=p["hidden_size"],input_chunk_length=p["input_chunk_length"],dropout=p["dropout"],training_length=500,**common)
    if name=="tide": return TiDEModel(input_chunk_length=p["input_chunk_length"],hidden_size=p["hidden_size"],dropout=p["dropout"],num_encoder_layers=p["num_encoder_layers"],num_decoder_layers=p["num_decoder_layers"],**common)
    if name=="dlinear": return DLinearModel(input_chunk_length=p["input_chunk_length"],**common)
    if name=="nlinear": return NLinearModel(input_chunk_length=p["input_chunk_length"],normalize=False,**common)
    raise ValueError(name)

def tune(name):
    def objective(trial):
        p={"input_chunk_length":trial.suggest_categorical("input_chunk_length",[168,336]),"batch_size":trial.suggest_categorical("batch_size",[16,32,64]),"lr":trial.suggest_float("lr",1e-5,1e-1,log=True)}
        if name in {"tft","tsmixer","deepar","tide"}: p.update(hidden_size=trial.suggest_categorical("hidden_size",[64,128,256]),dropout=trial.suggest_float("dropout",0.0,0.3))
        if name=="tft": p.update(num_attention_heads=trial.suggest_categorical("num_attention_heads",[2,4]),lstm_layers=trial.suggest_categorical("lstm_layers",[1,2]))
        if name=="tide": p.update(num_encoder_layers=trial.suggest_int("num_encoder_layers",1,3),num_decoder_layers=trial.suggest_int("num_decoder_layers",1,3))
        torch.manual_seed(SEED)
        m=build_model(name,p,f"{name}_tuning")
        m.fit(train_t,val_series=val_t,future_covariates=ctrain_t,val_future_covariates=cval_t,verbose=False)
        full_cov=[a.concatenate(b) for a,b in zip(ctrain_t,cval_t)]
        pred=m.predict(n=len(val_t[0]),series=train_t,future_covariates=full_cov)
        pred=target_scaler.inverse_transform(pred); truth=target_scaler.inverse_transform(val_t)
        return float(np.mean(rmse(truth,pred)))
    timeout=3600 if name in {"dlinear","nlinear"} else 5400
    study=optuna.create_study(direction="minimize")
    study.optimize(objective,n_trials=N_TRIALS,timeout=timeout,show_progress_bar=True)
    print("Best validation RMSE:",study.best_value); print("Best params:",study.best_params)
    return study.best_params

if LOAD_EXISTING:
    if not MODEL_PATH.exists(): raise FileNotFoundError(f"Checkpoint not found: {MODEL_PATH}. Put it there or set LOAD_EXISTING=False.")
    model=MODEL_CLASSES[MODEL].load(str(MODEL_PATH),weights_only=False)
else:
    best_params=tune(MODEL)
    model=build_model(MODEL,best_params,f"{MODEL}_wind")
    model.fit(train_t,val_series=val_t,future_covariates=ctrain_t,val_future_covariates=cval_t,verbose=True)
    if SAVE_MODEL: model.save(str(MODEL_PATH))


## 5. Main test result


In [ ]:
backtests=model.historical_forecasts(series=series_t,future_covariates=covs_t,start=test[0].start_time(),forecast_horizon=24,stride=24,num_samples=NUM_SAMPLES,last_points_only=False,retrain=True,verbose=True)
forecast_scaled=[concatenate(x) for x in backtests]
forecast=target_scaler.inverse_transform(forecast_scaled)
rows=[]
for ts,fc in zip(series,forecast):
    gt=ts.slice_intersect(fc); fc=fc.slice_intersect(gt)
    out=pd.DataFrame({"timestamp":gt.time_index,"target":gt.values().flatten()})
    for q in MODEL_QUANTILES: out[str(q)]=fc.quantile(q).values().flatten()
    out["prediction"]=out["0.5"]
    out["item_id"]=int(ts.static_covariates["item_id"].iloc[0])
    rows.append(out)
result=pd.concat(rows,ignore_index=True)

# Keep a common 15-quantile CSV schema. Unsupported model quantiles remain NaN.
for _q in OUTPUT_QUANTILES:
    if str(_q) not in result.columns:
        result[str(_q)] = np.nan
result.to_csv(OUTPUT_DIR/f"{MODEL}.csv",index=False)
print_metrics(result,QUANTILES,label=MODEL)

# Wind speed -> wind power evaluation
power_result=convert_wind_result_to_power(result)

# Keep a common 15-quantile CSV schema. Unsupported model quantiles remain NaN.
for _q in OUTPUT_QUANTILES:
    if str(_q) not in power_result.columns:
        power_result[str(_q)] = np.nan
power_result.to_csv(OUTPUT_DIR/f"{MODEL}_power.csv",index=False)
print_metrics(power_result,QUANTILES,label=f"{MODEL} [wind power]")
